### Lab 3.1: Batching and Regularization

In this lab you will learn how to set up a dataset to be processed in batches, rather than processing the entire dataset in each training iteration, and explore neural network regularization.

In [2]:
import numpy as np
import torch

In [3]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
adult = fetch_ucirepo(id=2) 
  
# data (as pandas dataframes) 
X = adult.data.features 
y = adult.data.targets 
  
# metadata 
print(adult.metadata) 
  
# variable information 
print(adult.variables)

{'uci_id': 2, 'name': 'Adult', 'repository_url': 'https://archive.ics.uci.edu/dataset/2/adult', 'data_url': 'https://archive.ics.uci.edu/static/public/2/data.csv', 'abstract': 'Predict whether annual income of an individual exceeds $50K/yr based on census data. Also known as "Census Income" dataset. ', 'area': 'Social Science', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 48842, 'num_features': 14, 'feature_types': ['Categorical', 'Integer'], 'demographics': ['Age', 'Income', 'Education Level', 'Other', 'Race', 'Sex'], 'target_col': ['income'], 'index_col': None, 'has_missing_values': 'yes', 'missing_values_symbol': 'NaN', 'year_of_dataset_creation': 1996, 'last_updated': 'Tue Sep 24 2024', 'dataset_doi': '10.24432/C5XW20', 'creators': ['Barry Becker', 'Ronny Kohavi'], 'intro_paper': None, 'additional_info': {'summary': "Extraction was done by Barry Becker from the 1994 Census database.  A set of reasonably clean records was extracted using the fol

In [4]:
X.columns

Index(['age', 'workclass', 'fnlwgt', 'education', 'education-num',
       'marital-status', 'occupation', 'relationship', 'race', 'sex',
       'capital-gain', 'capital-loss', 'hours-per-week', 'native-country'],
      dtype='str')

In [5]:
y = y['income'].map({'<=50K':0,'<=50K.':0,'>50K':1,'>50K.':1})

Here I remove the missing values from the features and labels.

In [6]:
bad = X.isna().any(axis=1)
X = X[~bad]
y = y[~bad]

Selecting only the numeric variables:

In [7]:
X = X[['age','fnlwgt','education-num','capital-gain','capital-loss','hours-per-week']]

In [8]:
y = y.values
X = X.values.astype('float64')

To make the learning algorithm work more smoothly, we we will subtract the mean and divide by the std. dev. of each feature.

Here `np.mean` calculates a mean, and `axis=0` tells NumPy to calculate the mean over the rows (calculate the mean of each column).

In [9]:
X -= np.mean(X,axis=0)
X /= np.std(X,axis=0)

Now we will convert our `X` and `y` arrays to torch Tensors.

In [10]:
X = torch.tensor(X).float()
y = torch.tensor(y).long()

### Exercises

1. Divide the data into train and test splits.
2. Create a neural network for this dataset.
3. Use `TensorDataset` and `DataLoader` to batch the dataset during training.  
4. Use `weight_decay` parameter to `optim.SGD` to introduce L2 regularization during training. Evaluate the effect of regularization on test set accuracy.

In [15]:
n = X.shape[0]
train_frac = 0.8
n_train = int(n * train_frac)

indices = torch.randperm(n)

train_idx = indices[:n_train]
test_idx = indices[n_train:]

X_train, y_train = X[train_idx], y[train_idx]
X_test, y_test = X[test_idx], y[test_idx]

nn = torch.nn.Sequential(
    torch.nn.Linear(X.shape[1], 100),
    torch.nn.ReLU(),
    torch.nn.Linear(100, 100),
    torch.nn.ReLU(),
    torch.nn.Linear(100, 100),
    torch.nn.ReLU(),
    torch.nn.Linear(100, 2),
)

train_dataset = torch.utils.data.TensorDataset(X_train, y_train)
test_dataset = torch.utils.data.TensorDataset(X_test, y_test)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=32, shuffle=False)

opt = torch.optim.SGD(nn.parameters(), lr=0.01, weight_decay=0.001)
loss_func = torch.nn.CrossEntropyLoss()

for epoch in range(20):
    for X_batch, y_batch in train_loader:
        opt.zero_grad()
        outputs = nn(X_batch)
        loss = loss_func(outputs, y_batch)
        loss.backward()
        opt.step()

nn.eval()
correct = 0
total = 0
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        outputs = nn(X_batch)
        preds = outputs.argmax(dim=1)
        correct += (preds == y_batch).sum().item()
        total += y_batch.size(0)
print(correct / total)


0.8226771653543307


Thank you Claude for this `for` loop

In [17]:
weight_decays = [0.0, 0.0001, 0.001, 0.01, 0.1]
results = {}

for wd in weight_decays:
    # fresh model every time so each trial starts from scratch
    model = torch.nn.Sequential(
        torch.nn.Linear(X.shape[1], 100),
        torch.nn.ReLU(),
        torch.nn.Linear(100, 100),
        torch.nn.ReLU(),
        torch.nn.Linear(100, 100),
        torch.nn.ReLU(),
        torch.nn.Linear(100, 2),
    )

    opt = torch.optim.SGD(model.parameters(), lr=0.01, weight_decay=wd)
    loss_func = torch.nn.CrossEntropyLoss()

    model.train()
    for epoch in range(100):
        for X_batch, y_batch in train_loader:
            opt.zero_grad()
            outputs = model(X_batch)
            loss = loss_func(outputs, y_batch)
            loss.backward()
            opt.step()

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            outputs = model(X_batch)
            preds = outputs.argmax(dim=1)
            correct += (preds == y_batch).sum().item()
            total += y_batch.size(0)

    accuracy = correct / total
    results[wd] = accuracy
    print(f"weight_decay={wd:<8} test accuracy={accuracy:.4f}")


weight_decay=0.0      test accuracy=0.8277
weight_decay=0.0001   test accuracy=0.8274
weight_decay=0.001    test accuracy=0.8264
weight_decay=0.01     test accuracy=0.8193
weight_decay=0.1      test accuracy=0.7595


The weight decay is having a negative effect on the test accuracy, perhaps because of the low epoch counts (at 30). I raised it and it had a similar effect at 100 epochs.